In [ ]:
%load_ext autoreload
%autoreload 2

import os
from src.constants import BASEDIR
os.chdir(BASEDIR)

from src.data.objects import Protein, ProteinDocument
from src.data.preprocessing import load_named_preprocessor
from src.data.proteingym import load_completions
from src.models.inference import InterleavedInverseFoldingPromptBuilder
from src.models.scoring import ProFamScorer
from src.models.utils import load_named_model

In [ ]:
DMS_FILENAME = "BLAT_ECOLX_Jacquier_2013.csv"
# TODO: check that filtered files always contain WT
UNIPROT_ID = "_".join(DMS_FILENAME.split("_")[:2])
MODEL_NAME = "twilight-snowflake"
MODEL_CONFIG = "llama_medium"

In [ ]:
completions, dms_df = load_completions(os.path.join(BASEDIR, f"data/example_data/ProteinGym/DMS_ProteinGym_substitutions/{DMS_FILENAME}"))

In [ ]:
preprocessor = load_named_preprocessor("parquet_structure", overrides=["structure_tokens_col=null"])

In [ ]:
preprocessor.transform_fns

In [ ]:
model = load_named_model(MODEL_CONFIG, overrides=["model.embed_coords=True"])

In [ ]:
prompt_builder = InterleavedInverseFoldingPromptBuilder(
    preprocessor=preprocessor,
    max_tokens=1536,
)

In [ ]:
protein = Protein.from_pdb(os.path.join(BASEDIR, f"data/example_data/ProteinGym/ProteinGym_AF2_structures/{UNIPROT_ID}.pdb"), bfactor_is_plddt=True)
protein.sequence

In [ ]:
scorer = ProFamScorer(
    "if_v1",
    model,
    prompt_builder=prompt_builder,
    document_token="[AFDB]",
    checkpoint_path=os.path.join(BASEDIR, f"outputs/inverse_folding/{MODEL_NAME}/checkpoints/last.ckpt"),
    bos_token="",
)
scores, prompt = scorer.score_completions(ProteinDocument.from_proteins([protein], representative_accession=protein.accession), completions)
scores


In [ ]:
prompt.sequences